# Adaptive Retrieval Routing

*The narrative companion to `adaptive_retrieval_routing.py`.* Every number below comes from that
module — this notebook imports it and never redefines the mathematics.

Run the reference implementation first; it must exit 0:

```
uv run --with numpy --with scipy --with scikit-learn python notebooks/adaptive-retrieval-routing/adaptive_retrieval_routing.py
```

**The question.** Selective generation ended with a gate that had two actions: emit the answer or
abstain. It can decline to answer, but it cannot try harder. This topic widens the action set — answer
from the query alone, retrieve once, or iterate — and asks which to spend on each query, deciding
*before* any retrieval fires.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import adaptive_retrieval_routing as R

c = R._corpus()
print(f"{c['n_queries']} queries, {c['n_docs']} documents, {c['K']} companies, dim {c['dim']}")
print("arms:", R.ARMS, "costs:", [R.ARM_COST[a] for a in R.ARMS])

## 1. Three arms, one answer model

The three arms are the **same imported answer model reading different evidence**, which is what makes
a quality difference attributable to the strategy rather than to the scorer. That also buys three
collapse anchors:

- `arm_none` is the zero-weight degenerate call — the document term vanishes and only the query
  remains;
- `arm_single` at unit weight **is** the imported `answer_posterior`;
- `arm_iterative(max_hops=1)` is byte-for-byte `arm_single`.

In [ ]:
from pmi_retrieval_value import answer_posterior

q = c["Q"][0]
d = c["docs"][R._top1(c, q)]

print("arm_single == imported answer_posterior :",
      np.max(np.abs(R.arm_single(c, q) - answer_posterior(q, d, c["protos"], tau=R.TAU_ANS))))
print("arm_iterative(1) == arm_single          :",
      np.max(np.abs(R.arm_iterative(c, q, max_hops=1) - R.arm_single(c, q))))

### The reversal that makes routing possible

Three query classes over one passage set. The compositional class is the one that matters: its answer
is a company a bridge filing **names** rather than **describes**, so a single retrieval cannot reach
it — the filing points mostly at the company it is about. Only reformulating off it recovers the
second direction.

In [ ]:
Qm = R.arm_quality(c)
klass = c["klass"]
print(f"{'class':8s} {'none':>8s} {'single':>8s} {'iter':>8s}   best        answers correct")
for k in ("known", "local", "bridge"):
    m = klass == k
    row = Qm[m].mean(axis=0)
    hits = [sum(int(np.argmax(R.ARM_FN[a](c, c["Q"][i]))) == int(c["truth"][i])
                for i in np.where(m)[0]) for a in R.ARMS]
    print(f"{k:8s} {row[0]:8.3f} {row[1]:8.3f} {row[2]:8.3f}   {R.ARMS[int(row.argmax())]:10s} "
          f"{hits}")

A single retrieval reaches the mentioned company **zero** times out of twenty-four; iterating reaches
it fourteen. And on the easy class all three arms are already correct every time — so routing there is
purely a question of what you are willing to pay, not of what you can get right.

## 2. Theorem 1 — the rule, and Chow's rule inside it

Minimizing the expected loss $c_{\mathrm{err}}(1 - Q_a) + c_a$ pointwise gives

$$\pi^\star(x) = \arg\max_a \big\{ \mathbb{E}[Q_a \mid \phi(x)] - \lambda c_a \big\}.$$

At $K=2$ over *emit* and *abstain* this is exactly Chow's rule — not approximately, and not only at the
published cost pair.

In [ ]:
from selective_generation_abstention import chow_decision, chow_threshold

scores = np.random.default_rng(0).uniform(0, 1, 400)
for c_err, c_abs in ((5.0, 1.0), (10.0, 1.0), (3.0, 2.5), (2.0, 1.0)):
    mine = R.chow_as_two_action(scores, c_err, c_abs)
    theirs = np.asarray(chow_decision(scores, chow_threshold(c_err, c_abs))).astype(int)
    print(f"c_err={c_err:5.1f} c_abs={c_abs:4.1f}  t*={chow_threshold(c_err, c_abs):.3f}  "
          f"disagreements={int((mine != theirs).sum())}")

## 3. Theorem 2 — the gain is a Jensen gap

$$\mathrm{Oracle} - \mathrm{BestFixed} = \mathbb{E}\big[\max_a U_a\big] - \max_a \mathbb{E}\big[U_a\big] \;\ge\; 0,$$

strictly positive **iff** the arms' advantage ordering varies across queries. The corollary is the
result worth carrying: if one arm is uniformly best, no router — however well estimated — beats simply
always using it.

In [ ]:
costs = np.array([R.ARM_COST[a] for a in R.ARMS])
print(f"{'lam':>6} {'oracle':>9} {'best fixed':>12} {'gap':>9}   per-query winner")
for lam in (0.0, 0.02, 0.05, 0.10, 0.20, 0.40):
    g = R.jensen_gap(Qm, lam, costs)
    print(f"{lam:6.2f} {g['oracle']:9.4f} {g['best_fixed']:12.4f} {g['gap']:+9.4f}   "
          f"{g['arm_win_counts']}  ({g['best_arm']})")

### The degenerate control

"The advantage ordering does not vary" means one shared per-query signal plus a fixed per-arm offset.
The argmax then cannot move, and the gap is identically zero — while the arms remain visibly
different from each other.

(Note what does *not* work: taking each query's maximum and adding a margin. Quality is a probability
mass, so that has to be clipped back into $[0,1]$, and the clip manufactures exact ties whose argmax
is decided by index order rather than by value.)

In [ ]:
for lam in (0.0, 0.02, 0.05, 0.10):
    g = R.degenerate_gap(Qm, lam, costs)
    print(f"lam={lam:5.2f}  gap={g['gap']:+.2e}  ordering varies: {g['argmax_varies']}  "
          f"{g['arm_win_counts']}")

## 4. Theorem 2' — the achievable region is a hull

Sweeping $\lambda$ traces deterministic policies; randomizing between adjacent ones fills the segment
between them, so the achievable set is their convex hull. Three frontiers sit one above another — and
**all three are scored on held-out queries**, because evaluating the router on the queries that fitted
it would inflate precisely the curve in question.

In [ ]:
fb = R.frontier_bundle(R.LAM_HEADLINE)
print(f"held-out queries: {fb['n_test']}, ridge chosen by CV inside the calibration half: {fb['ridge']:.0e}")
print()
print("best fixed arm (always iterate):  quality %.4f at cost %.2f" % (fb["fixed"][2][1], fb["fixed"][2][0]))
best_oracle = max(fb["oracle"], key=lambda p: p[1])
print("oracle, at its best quality:      quality %.4f at cost %.2f" % (best_oracle[1], best_oracle[0]))
print()
print("-> perfect routing buys BETTER answers for %.0f%% of the price." %
      (100 * best_oracle[0] / fb["fixed"][2][0]))

## 5. Proposition — what a real router actually collects

Substituting an *estimate* of each arm's quality bounds the excess loss by the summed $L_1$ estimation
error. So the router is no better than the calibration of the numbers it compares — the same caveat
Chow's rule carried one level down, where the threshold was only as good as the score it cut.

This is the honest half of the topic, and it is not flattering.

In [ ]:
print(f"{'lam':>6} {'in-sample':>11} {'held out':>10} {'oracle':>9} {'% of oracle':>12} "
      f"{'p':>7} {'ECE':>6}")
for lam in (0.02, 0.05, 0.10, 0.20):
    r = R.routing_report(lam)
    print(f"{lam:6.2f} {r['in_sample_gain']:+11.4f} {r['held_out_gain']:+10.4f} "
          f"{r['oracle_gain']:+9.4f} {100*r['held_out_fraction_of_oracle']:11.0f}% "
          f"{r['paired_p']:7.3f} {r['quality_model_ece']:6.3f}")

Read that table carefully, because three things in it are the point of the topic.

**In-sample exceeds held-out at every operating point.** A router tuned and scored on the same queries
looks better than it is. (This is also why the ridge penalty is selected by cross-validation *inside*
the calibration half — choosing it against the test half would be the same optimism one level up.)

**At $\lambda = 0.10$ the router loses.** The oracle gap there is $+0.054$, and the fitted router comes
in *below* always-retrieving-once. A gap that exists is not a gap you can collect.

**Only one of four gains is distinguishable from zero** at $n=36$. The evaluation track's thesis —
a metric is an estimator with a standard error, not a fact — applied to this topic's own headline.

## 6. Why the compositional class is the hard part

The router must identify the compositional class *before* retrieving, and the obvious features do not
separate it: entropy, nearest-prototype cosine and the top-two margin all place it **between** the
other two classes rather than apart from them. The reason is geometric — a compositional query's
closest competitor is a same-sector neighbour of the company it is *about*, not the company it
*names*, so the top-two margin is measuring the wrong pair.

In [ ]:
feats = R.router_features(c)
print(f"{'class':8s} {'H(prior)':>10s} {'max cos':>10s} {'top2 marg':>11s} {'cross-sector':>13s}")
for k in ("known", "local", "bridge"):
    m = klass == k
    print(f"{k:8s} " + " ".join(f"{feats[m, j].mean():>10.3f}" for j in range(4)))

Only the cross-sector maximum — the largest cosine to a prototype *outside the sector of the nearest
one* — separates the compositional class. With the first three features alone the router collapses to
always-retrieving-once at moderate $\lambda$ and gains nothing at all.

That feature was chosen against this corpus, which the topic's rigor flag states plainly. The claim
that does **not** depend on it is Theorem 2: the Jensen gap bounds what any router can win, and a real
one captures only part of it.

## 7. The full harness

Running the module as a script runs every check. Each one encodes a claim the topic makes.

In [ ]:
R._run_all()